# 01B: Exploratory Data Analysis

Characterises the fleet: site/circuit counts, geographic spread, export limits, key-table schemas, and single-day diagnostic plots.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
# sys.path.insert(0, str(pathlib.Path('lib').resolve()))

from shared.aws_config import aq, dread, s3_ls
from shared.ciccada_config import SA, SAI, AS4777, TABLES
from data_query.lib import explore_plots as ep
from data_query.lib import site_selection as ss
from data_query.lib import fleet_eda_diagnostics as fd

import pandas as pd
import numpy as np

## Sites and circuits

In [ ]:
# -----------------------------------------------------------------------------
# Assumptions:
# sites  = physical locations (one row per address)
# circuits = monitoring points within a site (one row per phase/device)
# is_pv = True means it's a solar PV circuit (not load, battery, etc.)
# -----------------------------------------------------------------------------

site_count = aq("SELECT count(*) AS n_sites FROM sites", database=SAI)
print("Total sites:", site_count["n_sites"].iloc[0])

In [ ]:
circuit_counts = aq("""
    SELECT
        is_pv,
        count(*)          AS n_circuits,
        count(DISTINCT site_id) AS n_sites
    FROM circuits
    GROUP BY is_pv
    ORDER BY is_pv DESC
""", 
database=SAI)
circuit_counts

# is_pv=True rows are what the telemetry analysis is built on.
# The values below double count sites because many sites have both PV and non-PV circuits.

In [ ]:
# How many circuits per site?
circuits_per_site = aq("""
    SELECT
        circuit_count,
        count(*) AS n_sites
    FROM (
        SELECT site_id, count(*) AS circuit_count
        FROM circuits
        WHERE is_pv = True
        GROUP BY site_id
    )
    GROUP BY circuit_count
    ORDER BY circuit_count
""", database=SAI)
circuits_per_site

## Geographic spread

In [ ]:
# -----------------------------------------------------------------------------
# Geographic spread — use meta_up23c, not sites
# -----------------------------------------------------------------------------

states = aq("""
    SELECT state, count(*) AS n_sites
    FROM meta_up23c
    GROUP BY state
    ORDER BY n_sites DESC
""", database=SAI)
states

In [ ]:
# -----------------------------------------------------------------------------
# Use partition_lookup (tiny table) rather than querying ts directly.
# Reading min/max timestamps from a billions-row table is expensive;
# the lookup table gives you the answer for free.
# -----------------------------------------------------------------------------

partitions = aq("SELECT * FROM partition_lookup ORDER BY year, month", database=SA)
partitions
# Each row = one (year, month) partition that exists in the ts table.
# The first and last rows tell you the data window.

In [ ]:
unique_months = partitions[['year', 'month']].drop_duplicates().sort_values(['year', 'month'])

print("Data covers:")
print(f"  From: {unique_months['year'].iloc[0]}-{str(unique_months['month'].iloc[0]).zfill(2)}")
print(f"  To:   {unique_months['year'].iloc[-1]}-{str(unique_months['month'].iloc[-1]).zfill(2)}")
print(f"  Total months: {len(unique_months)}")

## Export limits (flex_export_detected)

In [ ]:
aq('''
SELECT flex_export_detected, count(DISTINCT site_id) AS n_sites
FROM meta_up23c
WHERE is_pv = True
GROUP BY flex_export_detected
''', database='solar_analytics_iceberg')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# flex_export_detected diagnostic checks
# ═══════════════════════════════════════════════════════════════════════════════

#  Check 1: Where are the flagged sites?
flex_by_dnsp = aq("""
    SELECT
        flex_export_detected,
        state,
        dnsp_name,
        count(DISTINCT site_id)  AS n_sites,
        round(avg(ac_capacity_kw), 1) AS avg_ac_kw,
        round(avg(export_limit_kw), 1) AS avg_export_limit_kw
    FROM meta_up23c
    WHERE is_pv = True
    GROUP BY flex_export_detected, state, dnsp_name
    ORDER BY flex_export_detected DESC, n_sites DESC
""", database='solar_analytics_iceberg')

print("Flex-export sites by state and DNSP:")
print(flex_by_dnsp.to_string(index=False))

# Check 2: Do flagged sites have explicit export limits?
# If export_limit_kw is set AND is less than ac_capacity_kw, 
# that's a signal the site really is constrained.

flex_export_limits = aq("""
    SELECT
        flex_export_detected,
        count(DISTINCT site_id) AS n_sites,
        sum(CASE WHEN export_limit_kw IS NOT NULL THEN 1 ELSE 0 END) AS has_export_limit,
        sum(CASE WHEN export_limit_kw IS NOT NULL
                  AND export_limit_kw < ac_capacity_kw THEN 1 ELSE 0 END)
            AS export_limit_below_nameplate
    FROM (
        SELECT DISTINCT site_id, ac_capacity_kw, export_limit_kw, flex_export_detected
        FROM meta_up23c
        WHERE is_pv = True
    )
    GROUP BY flex_export_detected
""", database='solar_analytics_iceberg')

print("\nExport limit breakdown:")
print(flex_export_limits.to_string(index=False))

## Schema of the key tables

In [ ]:
# Schema discovery for all Iceberg tables.

iceberg_tables = [
    # stage 0
    "ts",
    "meta_up23c",
    # Stage 1
    "structured_data",
    # Stage 1 rebuilds
    "structured_data_v2_flex_included",
    "all_uncurtailedpv_v2_flex_included",
    # Stage 2 rebuilds
    "conformance_voltvar_v2_flex_included",
    "conformance_voltwatt_v2_flex_included",
    "conformance_voltwattghi_v2_flex_included",
    # not rebuilt. still legacy
    "conformance_antiisland",
    "conformance_sust_op_3w",
]

schemas = {}
for t in iceberg_tables:
    try:
        row = aq(f"SELECT * FROM {t} LIMIT 1", database=SAI)
        schemas[t] = pd.DataFrame({
            "column": row.columns.tolist(),
            "dtype":  [str(d) for d in row.dtypes.tolist()],
        })
        print(f"OK  {t:32s} {len(row.columns)} columns")
    except Exception as e:
        print(f"ERR {t:32s} {e}")

In [ ]:
# Open up schemas here:
schemas['structured_data_v2_flex_included']

In [ ]:
# Open up schemas here:
schemas['all_uncurtailedpv_v2_flex_included']

In [ ]:
# Open up schemas here:
schemas['ts']

In [ ]:
# Open up schemas here:
schemas['meta_up23c']

In [ ]:
# Quick look at some data:
pd.set_option('display.max_columns', None)
aq("SELECT * FROM structured_data_v2_flex_included WHERE year = 2024 AND month = 1 LIMIT 5", database=SAI)

## Circuit bands

In [ ]:
# Circuit count banded: makes the 1-to-60 range more interpretable
circuit_profile = aq("""
    SELECT
        CASE
            WHEN circuit_count = 1  THEN '1  - single phase'
            WHEN circuit_count = 2  THEN '2  - split or data gap'
            WHEN circuit_count = 3  THEN '3  - three phase'
            WHEN circuit_count <= 6 THEN '4-6 - multi-array or battery'
            ELSE                         '7+  - commercial / large site'
        END AS circuit_profile,
        count(*) AS n_sites,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM (
        SELECT site_id, count(*) AS circuit_count
        FROM circuits
        WHERE is_pv = True
        GROUP BY site_id
    )
    GROUP BY 1
    ORDER BY min(circuit_count)
""", database=SAI)
circuit_profile

## Fleet characteristics

In [ ]:
# -----------------------------------------------------------------------------
# Site summary four separate queries, one per count
# -----------------------------------------------------------------------------

n_sites         = aq("SELECT count(DISTINCT site_id) AS n FROM sites",       database=SAI)["n"].iloc[0]
n_meta_up23c    = aq("SELECT count(DISTINCT site_id) AS n FROM meta_up23c",  database=SAI)["n"].iloc[0]
n_pv_circuits   = aq("SELECT count(DISTINCT site_id) AS n FROM circuits WHERE is_pv = True", database=SAI)["n"].iloc[0]
#n_single_inv    = aq("SELECT count(*) AS n FROM meta_single_inverters",       database=SA) ["n"].iloc[0]

site_summary = pd.DataFrame([{
    "total_in_sites_table":       n_sites,
    "sites_in_meta_up23c":        n_meta_up23c,
    "sites_with_pv_circuit":      n_pv_circuits
#    "meta_single_inverters_rows": n_single_inv,
}])
site_summary

## Single-day diagnostic plots

Pick a site + date, pull one day, convert to AEST, and plot. Two views:
- `plot_operational`: Volt-Watt + Volt-VAr response
- `plot_protective`: sustained-operation + anti-islanding (over/under-voltage)

In [ ]:
# =============================================================================
# Select a response mode and pull a ranked site list
# =============================================================================
# RESPONSE_MODE : "voltwatt" | "voltvar" | "sust_op" | "sust_op_3w" | "antiisland"
# BEHAVIOUR     : "nonconforming" | "conforming"
# PLOT_TYPE     : "operational" | "protective"
# YEAR / MONTH  : MONTH=None ranks over the full year
#
# All conformance tables live in solar_analytics_iceberg (SAI).
# Ranking metric per mode:
#   voltwatt   — nonconformance_voltwatt_count
#   voltvar    — nonconformance_voltvar_red_count  (severe Q deviation only)
#                extra breakdown columns also pulled for context
#   sust_op    — nonconformance_sust_op_count
#   sust_op_3w — nonconformance_sust_op_3w_count
#   antiisland — nonconformance_antiisland_count
#
# NOTE on voltvar category name swap (pipeline bug from inherited code, old database):
#   q_minor_deviation  = 10–90% band  (larger shortfall — counterintuitively named)
#   q_major_deficit    = 90–110% band (near-miss   — counterintuitively named)
# =============================================================================

# =============================================================================
#### Q_impact = signed abs(Q_kvar) / abs(nearest permitted final band edge) ####
#         Q_impact < -10%   = Adverse
# -10% <= Q_impact <= +10%  = Inactive
# +10% <  Q_impact <= +90%  = Major deficit
# +90% <= Q_impact <= +110% = Minor deviation
# +110% < Q_impact          = Major surplus

mode = "sust_op_3w"

# Rank sites for a mechanism
ranked = ss.rank_sites(
    mode=mode, 
    aq_func=aq, 
    behaviour="conforming",
    year=2025, 
    min_days=20,
    n_results=100)
ranked

In [ ]:
ranked[:30]

In [ ]:
# 2. Pull that site nameplate/year/month/polarity
SITE_ID = ranked["site_id"].iloc[3]
# SITE_ID = 1033373679
# SITE_ID = 2134086510
# SITE_ID = 465008538
# SITE_ID = 276149647
SITE_ID = 1644145371

SITE_ID = 931508817

SITE_ID = 465008538

df, info = ss.pull_site_telemetry(SITE_ID, mode, aq, year=2025, month=2)

In [ ]:
# 3. Which day to plot?
ss.suggest_days(df, mode)

### Operational plots

In [ ]:
# 4. Plot pick a date that's actually in df
# df onlys pulls one month, so pick a date from the printed list above or from suggest_days().
print("Month pulled:", info["plot_year"], info["plot_month"])
print("Dates available:", sorted(df["t_stamp_aest"].dt.date.unique())[:5], "...")

ZOOM_DATE = "2025-02-15"
df_day = df[df["t_stamp_aest"].dt.date == pd.Timestamp(ZOOM_DATE).date()].copy()
assert len(df_day) > 0, f"No rows for {ZOOM_DATE}. Pick from the dates printed above or from suggest_days()."

ep.plot_operational(df_day, info["site_id"], info["ac_capacity_kw"], ZOOM_DATE, AS4777)

In [ ]:
df_day

#### Isolated Volt-Watt

In [ ]:
site_year = aq(f"""
    SELECT
        site_id,
        sum(nonconformance_voltwattghi_count) AS nonconf_count,
        sum(assessable_count)                 AS response_supported_count
    FROM {TABLES['conformance_voltwattghi']}
    WHERE year IN (2024, 2025)
    GROUP BY site_id
""", database=SAI)

MIN_INTERVALS = 50   # require decent evidence, not one lucky interval
site_year["nonconf_frac"] = site_year["nonconf_count"] / site_year["response_supported_count"]
conformant_sites = site_year[
    (site_year["response_supported_count"] >= MIN_INTERVALS) &
    (site_year["nonconf_frac"] < 0.10)
].sort_values("response_supported_count", ascending=False)

print(f"{len(conformant_sites)} conformant sites with >= {MIN_INTERVALS} response-supported intervals")
conformant_sites.head(20)

In [ ]:
site_ids_sql = ", ".join(str(int(s)) for s in conformant_sites["site_id"].head(50))

candidate_days = aq(f"""
    SELECT
        c.site_id, c.year, c.month, c.day,
        m.state, m.dnsp_name, m.ac_capacity_kw, m.manufacturer,
        c.P_kW_sum, c.total_count, c.assessable_count,
        c.nonconformance_voltwattghi_count
    FROM {TABLES['conformance_voltwattghi']} c
    JOIN (
        SELECT DISTINCT site_id, state, dnsp_name, ac_capacity_kw, manufacturer
        FROM meta_up23c WHERE is_pv = True
    ) m ON m.site_id = c.site_id
    WHERE c.day_night = 'day'
      AND c.site_id IN ({site_ids_sql})
      AND c.assessable_count >= 5
    ORDER BY c.nonconformance_voltwattghi_count ASC, c.assessable_count DESC
    LIMIT 30
""", database=SAI)

candidate_days

In [ ]:
row = candidate_days.iloc[12]
SITE_ID   = int(row["site_id"])
ZOOM_DATE = f'{int(row["year"])}-{int(row["month"]):02d}-{int(row["day"]):02d}'
print(SITE_ID, ZOOM_DATE, "| assessable:", row["assessable_count"], "| NC that day:", row["nonconformance_voltwattghi_count"])

df, info = ss.pull_site_telemetry(SITE_ID, "voltwatt", aq, year=int(row["year"]), month=int(row["month"]))
df_day = df[df["t_stamp_aest"].dt.date == pd.Timestamp(ZOOM_DATE).date()].copy()

ep.plot_voltwatt_only(df_day, info["site_id"], info["ac_capacity_kw"], ZOOM_DATE, AS4777)

In [ ]:
from data_query.lib import voltwatt_curtailment as vwc

TARGET_SITE = SITE_ID 
TARGET_DATE = ZOOM_DATE

day_df = vwc.fetch_voltwatt_day_data(
    aq, site_id=TARGET_SITE, date_str=TARGET_DATE,
    database=SAI,
    all_uncurtailedpv_table=TABLES["all_uncurtailedpv"],
)

vwc.plot_voltwatt_curtailment_day(
    day_df,
    site_id=TARGET_SITE,
    date_str=TARGET_DATE,
    as4777=AS4777,
    redact_site=True,
#    interpolate=True
);

In [ ]:
df = day_df.copy()
df["t"] = (pd.to_datetime(df["t_stamp"]).dt.tz_localize("UTC")
           .dt.tz_convert(vwc.FIXED_OFFSET).dt.tz_localize(None))

S = df["rating_capacity"].median()
from shared.as4777_curves import vw_max_p, add_tol_kw
df["P_ceiling_kW"] = df["V"].apply(lambda v: add_tol_kw(vw_max_p(v, S), S, AS4777["TOL_FRAC"]))

eligible = df["V"] > AS4777["VW"]["V1"]
gap = eligible & ~(df["P_kW"] > df["P_ceiling_kW"]) & ~(
    df["P_potential_kW"].notna() & (df["P_potential_kW"] > df["P_ceiling_kW"])
)
df.loc[gap, ["t", "V", "P_kW", "P_ceiling_kW", "P_potential_kW"]]

In [ ]:
site_ids_sql = ", ".join(str(int(s)) for s in conformant_sites["site_id"].head(200))

mixed_days = aq(f"""
    SELECT
        c.site_id, c.year, c.month, c.day,
        m.state, m.dnsp_name, m.ac_capacity_kw, m.manufacturer,
        c.total_count, c.assessable_count,
        c.nonconformance_voltwattghi_count,
        c.curtailment_voltwattghi_count
    FROM {TABLES['conformance_voltwattghi']} c
    JOIN (
        SELECT DISTINCT site_id, state, dnsp_name, ac_capacity_kw, manufacturer
        FROM meta_up23c WHERE is_pv = True
    ) m ON m.site_id = c.site_id
    WHERE c.day_night = 'day'
      AND c.site_id IN ({site_ids_sql})
      AND c.nonconformance_voltwattghi_count > 0      -- some non-conformant intervals
      AND c.curtailment_voltwattghi_count > 0         -- AND some genuine curtailment
      AND c.assessable_count >= 20                    -- decent exposure for a clean figure
    ORDER BY c.assessable_count DESC
    LIMIT 30
""", database=SAI)

mixed_days

In [ ]:
row = mixed_days.iloc[0]
SITE_ID   = int(row["site_id"])
ZOOM_DATE = f'{int(row["year"])}-{int(row["month"]):02d}-{int(row["day"]):02d}'
print(SITE_ID, ZOOM_DATE, "| assessable:", row["assessable_count"], "| NC that day:", row["nonconformance_voltwattghi_count"])

df, info = ss.pull_site_telemetry(SITE_ID, "voltwatt", aq, year=int(row["year"]), month=int(row["month"]))
df_day = df[df["t_stamp_aest"].dt.date == pd.Timestamp(ZOOM_DATE).date()].copy()

ep.plot_voltwatt_only(df_day, info["site_id"], info["ac_capacity_kw"], ZOOM_DATE, AS4777)

#### Monthly Volt-VAr

In [ ]:
# uses the month that pull_site_telemetry resolved
scatter_df = ss.pull_month_scatter(
    info["site_id"], 
    info["plot_year"],
    info["plot_month"], 
    aq
)
ep.plot_vvar_month_scatter(
    scatter_df, 
    info["site_id"], 
    info["ac_capacity_kw"],
    f"{info['plot_year']}-{info['plot_month']:02d}",
    manufacturer=info["manufacturer"]
)

### Operational diagnostics

In [ ]:
stored_verdict = fd.fetch_stored_day_verdict(
    SITE_ID,
    ZOOM_DATE,
    aq,
    SAI,
)

recomputed = fd.recompute_vvar_day(
    SITE_ID,
    ZOOM_DATE,
    aq,
    SAI,
)

interval_summary, bucket_summary = (
    fd.summarise_recomputed_day(recomputed)
)

display(stored_verdict)
display(interval_summary)
display(recomputed)
display(bucket_summary)

In [ ]:
timestamp_status = recomputed[
    [
        "t_stamp_aest",
        "day_night",
        "V",
        "P_kW",
        "P_fraction_of_rated_proxy",
        "Q_kvar",
        "Q_voltvar",
        "Q_max_final",
        "Q_min_final",
        "capability_assessable",
        "outside_band",
        "Q_impact",
        "status",
    ]
]

display(timestamp_status)

In [ ]:
status_summary = (
    recomputed
    .groupby(["day_night", "status"], dropna=False)
    .size()
    .rename("intervals")
    .reset_index()
)

display(status_summary)

In [ ]:
high_voltage_check = fd.fetch_low_power_high_voltage(
    SITE_ID,
    info["plot_year"],
    info["plot_month"],
    aq,
    SAI,
)

display(high_voltage_check)

In [ ]:
465008538

In [ ]:
stored_vw = fd.fetch_stored_voltwatt_day_verdict(
    SITE_ID,
    ZOOM_DATE,
    aq,
    SAI,
)

if stored_vw.empty:
    raise ValueError(
        f"No stored Volt-Watt result for site {SITE_ID}, {ZOOM_DATE}"
    )

# Recompute using the provenance recorded in the stored table.
vw_provenance = (
    stored_vw[
        [
            "rating_basis",
            "voltage_aggregation",
            "flex_selection",
        ]
    ]
    .drop_duplicates()
)

if len(vw_provenance) != 1:
    raise ValueError(
        "Expected exactly one Volt-Watt provenance configuration "
        f"for this site-day; found {len(vw_provenance)}."
    )

vw_options = vw_provenance.iloc[0].to_dict()

recomputed_vw = fd.recompute_voltwatt_day(
    SITE_ID,
    ZOOM_DATE,
    aq,
    SAI,
    **vw_options,
)

vw_timestamp_status = recomputed_vw[
    [
        "t_stamp_aest",
        "day_night",
        "V",
        "P_kW",
        "rating_kW",
        "P_limit_curve_kW",
        "tolerance_kW",
        "P_limit_with_tolerance_kW",
        "margin_to_limit_kW",
        "P_excess_kW",
        "exposed",
        "status",
    ]
]

vw_status_summary = (
    recomputed_vw
    .groupby(["day_night", "status"], dropna=False)
    .size()
    .rename("intervals")
    .reset_index()
)

display(stored_vw)
display(vw_timestamp_status)
display(vw_status_summary)

In [ ]:
# vw_timestamp_status.to_csv("vw_timestamp_status.csv", index=False)

### Protective

In [ ]:
ep.plot_protective(df_day, info["site_id"], info["ac_capacity_kw"], ZOOM_DATE)